# APTOS 2019 Blindness Detection - Colab Training Notebook
This notebook downloads the dataset via Kaggle, sets up the Hybrid CNN + ViT model, and trains it on Colab GPUs.

In [ ]:
!pip install -q kaggle
!pip install -q torch torchvision pandas numpy Pillow tqdm

### Step 1: Set Kaggle API Credentials
We have embedded the API token you provided. All you need to do is replace the username with your exact Kaggle username.

In [ ]:
import os
import re

# IMPORTANT: Replace 'YOUR_KAGGLE_USERNAME' below with your actual username (all lowercase, no spaces)
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME' 
os.environ['KAGGLE_KEY'] = 'KGAT_6da9e525a6f6130b753d55e1c8ea556e'

print("Credentials securely set in environment variables!")

### Step 2: Download Dataset
This pulls the dataset directly into the Colab environment.

In [ ]:
!kaggle competitions download -c aptos2019-blindness-detection

!mkdir -p /content/aptos2019
!unzip -q aptos2019-blindness-detection.zip -d /content/aptos2019/
!rm aptos2019-blindness-detection.zip

print("Dataset Extracted to /content/aptos2019/")

### Step 3: Train the Model

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from PIL import Image
import gc
from tqdm import tqdm

# ================= Configuration =================
class Config:
    SEED = 42
    BATCH_SIZE = 16  
    EPOCHS = 15
    LEARNING_RATE = 1e-4
    NUM_CLASSES = 5
    IMAGE_SIZE = 224
    
    # Colab paths
    DATA_DIR = '/content/aptos2019/train_images' 
    CSV_PATH = '/content/aptos2019/train.csv'
    MODEL_SAVE_PATH = '/content/dr_hybrid_model.pth'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

# ================= Model Architecture =================
class HybridDRClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super(HybridDRClassifier, self).__init__()
        self.vit = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        vit_out_dim = self.vit.heads.head.in_features
        self.vit.heads = nn.Identity()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Linear(vit_out_dim + 64, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, img_vit, img_cnn):
        f_vit = self.vit(img_vit)
        f_cnn = self.cnn(img_cnn)
        f_cnn = f_cnn.view(f_cnn.size(0), -1) 
        f_fused = torch.cat((f_vit, f_cnn), dim=1)
        logits = self.fc(f_fused)
        return logits, f_fused

# ================= Loss Function =================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce = nn.CrossEntropyLoss(reduction='none')
    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

# ================= Data Loader =================
class DRDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.df['image_path'] = self.df['id_code'].apply(lambda x: os.path.join(self.img_dir, f"{x}.png"))
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['image_path']
        label = self.df.iloc[idx]['diagnosis']
        try: image = Image.open(img_path).convert("RGB")
        except Exception: image = Image.new('RGB', (Config.IMAGE_SIZE, Config.IMAGE_SIZE))
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

seed_everything(Config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = DRDataset(Config.CSV_PATH, Config.DATA_DIR, transform=train_transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = HybridDRClassifier(num_classes=Config.NUM_CLASSES).to(device)
for param in model.vit.parameters(): param.requires_grad = False
criterion = FocalLoss()
optimizer = optim.Adam(model.parameters(), lr=Config.LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

best_val_loss = float('inf')
print("Starting Training on Colab...")
for epoch in range(Config.EPOCHS):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(images, images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix({'loss': loss.item()})
    
    train_loss /= len(train_loader.dataset)
    train_acc = correct / total
    
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits, _ = model(images, images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = logits.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_loss /= len(val_loader.dataset)
    val_acc = val_correct / val_total
    scheduler.step(val_loss)
    print(f"Epoch {epoch+1} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), Config.MODEL_SAVE_PATH)
    gc.collect(); torch.cuda.empty_cache()
print("Training Complete!")

### Step 4: Download The Trained Model

In [ ]:
from google.colab import files
files.download('/content/dr_hybrid_model.pth')